Library imports

In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

In [5]:
DATA_DIR = Path(r"C:\Users\Harshini J\Engineering\Projects\Nimisha Ma'am - Project\mamba-dti\data\raw\human_random")

In [6]:
for file in sorted(DATA_DIR.iterdir()):
    print(file.name)

test
train
valid


In [13]:
train_dir = DATA_DIR / "train"

for csv_file in sorted(train_dir.glob("*.csv")):
    print(csv_file.name)
    df = pd.read_csv(csv_file)
    print("Shape:", df.shape)
    print()
    print(df.head())
    print()
    print(df.info())

samples.csv
Shape: (5382, 3)

   smiles  sequence  interactions
0       0         0             0
1       1         1             0
2       2         2             1
3       3         3             0
4       4         4             0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5382 entries, 0 to 5381
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   smiles        5382 non-null   int64
 1   sequence      5382 non-null   int64
 2   interactions  5382 non-null   int64
dtypes: int64(3)
memory usage: 126.3 KB
None
sequence.csv
Shape: (1840, 2)

   index                                           sequence
0      0  MRLSKTLVDMDMADYSAALDPAYTTLEFENVQVLTMGNDTSPSEGT...
1      1  MAQKGQLSDDEKFLFVDKNFINSPVAQADWAAKRLVWVPSEKQGFE...
2      2  MPAENSPAPAYKVSSHGGDSGLDGLGGPGVQLGSPDKKKRKANTQG...
3      3  MSSSCSGLSRVLVAVATALVSASSPCPQAWGPPGVQYGQPGRSVKL...
4      4  MAEAHQAVAFQFTVTPDGIDLRLSHEALRQIYLSGLHSWKKKFIRF...

<class 'pandas.co

In [14]:
for csv_file in sorted(train_dir.glob("*.csv")):
    df = pd.read_csv(csv_file)
    print(csv_file.name)
    print(df.columns.tolist())
    print()

samples.csv
['smiles', 'sequence', 'interactions']

sequence.csv
['index', 'sequence']

smiles.csv
['index', 'smiles']



Merging the lookup tables for better exploration

In [15]:
samples = pd.read_csv(train_dir / "samples.csv")
smiles = pd.read_csv(train_dir / "smiles.csv")
sequence = pd.read_csv(train_dir / "sequence.csv")

In [16]:
smiles = smiles.rename(columns={
    "index": "smiles_id",
    "smiles": "smiles_string"
})

sequence = sequence.rename(columns={
    "index": "sequence_id"
})

In [17]:
samples = samples.rename(columns={
    "smiles": "smiles_id",
    "sequence": "sequence_id",
    "interactions": "label"
})

In [18]:
df = (
    samples
    .merge(smiles, on="smiles_id")
    .merge(sequence, on="sequence_id")
)

In [19]:
print(df.shape)
print(df.head())

(5382, 5)
   smiles_id  sequence_id  label  \
0          0            0      0   
1          1            1      0   
2          2            2      1   
3          3            3      0   
4          4            4      0   

                                       smiles_string  \
0           COC1[C@H]([C@H]([C@@H]([C@H](O1)CO)O)O)O   
1                                    C1=CC=C(C=C1)Cl   
2                                               [Zn]   
3  C[C@@H](C(=O)N[C@H](CCC(=O)O)C(=O)O)NC(=O)[C@@...   
4      C1[C@H](O[C@H]([C@H]1F)N2C=NC3=C2N=CN=C3NN)CO   

                                            sequence  
0  MRLSKTLVDMDMADYSAALDPAYTTLEFENVQVLTMGNDTSPSEGT...  
1  MAQKGQLSDDEKFLFVDKNFINSPVAQADWAAKRLVWVPSEKQGFE...  
2  MPAENSPAPAYKVSSHGGDSGLDGLGGPGVQLGSPDKKKRKANTQG...  
3  MSSSCSGLSRVLVAVATALVSASSPCPQAWGPPGVQYGQPGRSVKL...  
4  MAEAHQAVAFQFTVTPDGIDLRLSHEALRQIYLSGLHSWKKKFIRF...  


In [20]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5382 entries, 0 to 5381
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   smiles_id      5382 non-null   int64 
 1   sequence_id    5382 non-null   int64 
 2   label          5382 non-null   int64 
 3   smiles_string  5382 non-null   object
 4   sequence       5382 non-null   object
dtypes: int64(3), object(2)
memory usage: 210.4+ KB


In [21]:
df.isnull().sum()

smiles_id        0
sequence_id      0
label            0
smiles_string    0
sequence         0
dtype: int64

In [22]:
df.duplicated().sum()

np.int64(517)

In [23]:
df.duplicated(subset=["smiles_id", "sequence_id"]).sum()

np.int64(517)

In [24]:
df["label"].value_counts()

label
1    2709
0    2673
Name: count, dtype: int64

In [25]:
df["label"].value_counts(normalize=True)

label
1    0.503344
0    0.496656
Name: proportion, dtype: float64

In [26]:
df["protein_length"] = df["sequence"].str.len()

In [27]:
df["smiles_length"] = df["smiles_string"].str.len()

In [28]:
df[["protein_length", "smiles_length"]].describe()

,protein_length,smiles_length
count,5382.000000,5382.000000
mean,621.085470,46.890004
std,514.128435,41.901281
min,39.000000,2.000000
25%,322.000000,17.000000
50%,466.000000,41.000000
75%,725.000000,61.000000
max,5038.000000,420.000000


In [29]:
conflicts = (
    df.groupby(["smiles_id", "sequence_id"])["label"]
      .nunique()
)

conflicts = conflicts[conflicts > 1]

print("Conflicting pairs:", len(conflicts))

Conflicting pairs: 0


In [30]:
print(df["smiles_id"].nunique())
print(df["sequence_id"].nunique())

2356
1840


In [31]:
duplicates = df[df.duplicated(keep=False)].sort_values(
    by=["smiles_id", "sequence_id"]
)

duplicates.head(20)

,smiles_id,sequence_id,label,smiles_string,sequence,protein_length,smiles_length
2,2,2,1,[Zn],MPAENSPAPAYKVSSHGGDSGLDGLGGPGVQLGSPDKKKRKANTQG...,419,4
989,2,2,1,[Zn],MPAENSPAPAYKVSSHGGDSGLDGLGGPGVQLGSPDKKKRKANTQG...,419,4
10,2,10,1,[Zn],MATAGNPWGWFLGYLILGVAGSLVSGSCSQIINGEDCSPHSQPWQA...,254,4
3428,2,10,1,[Zn],MATAGNPWGWFLGYLILGVAGSLVSGSCSQIINGEDCSPHSQPWQA...,254,4
3791,2,10,1,[Zn],MATAGNPWGWFLGYLILGVAGSLVSGSCSQIINGEDCSPHSQPWQA...,254,4
4624,2,10,1,[Zn],MATAGNPWGWFLGYLILGVAGSLVSGSCSQIINGEDCSPHSQPWQA...,254,4
4707,2,10,1,[Zn],MATAGNPWGWFLGYLILGVAGSLVSGSCSQIINGEDCSPHSQPWQA...,254,4
4994,2,10,1,[Zn],MATAGNPWGWFLGYLILGVAGSLVSGSCSQIINGEDCSPHSQPWQA...,254,4
1212,2,26,1,[Zn],MVVMNSLRVILQASPGKLLWRKFQIPRFMPARPCSLYTCTYKTRNR...,317,4
2291,2,26,1,[Zn],MVVMNSLRVILQASPGKLLWRKFQIPRFMPARPCSLYTCTYKTRNR...,317,4


In [32]:
duplicate_groups = (
    df.groupby(["smiles_id", "sequence_id"])
      .size()
      .sort_values(ascending=False)
)

duplicate_groups.head(20)

smiles_id  sequence_id
81         658            15
107        42             13
93         42             11
405        15              9
330        15              7
66         15              7
15         15              7
55         15              7
73         15              6
2          10              6
167        15              6
137        42              6
464        15              6
218        15              6
825        42              5
81         42              5
307        42              5
608        15              5
390        15              5
402        15              5
dtype: int64

In [33]:
samples = pd.read_csv(train_dir / "samples.csv")

print("Raw samples:", len(samples))
print(
    "Duplicate (smiles, sequence) pairs:",
    samples.duplicated(subset=["smiles", "sequence"]).sum()
)

Raw samples: 5382
Duplicate (smiles, sequence) pairs: 517


In [34]:
unique_pairs = (
    df[["smiles_id", "sequence_id"]]
    .drop_duplicates()
)

print("Total rows      :", len(df))
print("Unique pairs    :", len(unique_pairs))
print("Duplicate rows  :", len(df) - len(unique_pairs))

Total rows      : 5382
Unique pairs    : 4865
Duplicate rows  : 517


In [35]:
train_seq = pd.read_csv(DATA_DIR / "train" / "sequence.csv")
valid_seq = pd.read_csv(DATA_DIR / "valid" / "sequence.csv")
test_seq  = pd.read_csv(DATA_DIR / "test" / "sequence.csv")

In [36]:
print(f"Train proteins : {len(train_seq)}")
print(f"Valid proteins : {len(valid_seq)}")
print(f"Test proteins  : {len(test_seq)}")

Train proteins : 1840
Valid proteins : 469
Test proteins  : 465


In [37]:
train_set = set(train_seq["sequence"])
valid_set = set(valid_seq["sequence"])
test_set  = set(test_seq["sequence"])

In [38]:
print(f"Unique train proteins : {len(train_set)}")
print(f"Unique valid proteins : {len(valid_set)}")
print(f"Unique test proteins  : {len(test_set)}")

Unique train proteins : 1840
Unique valid proteins : 469
Unique test proteins  : 465


In [39]:
all_proteins = train_set | valid_set | test_set

print(f"Total unique proteins across all splits: {len(all_proteins)}")

Total unique proteins across all splits: 2001


In [40]:
print("Train ∩ Valid :", len(train_set & valid_set))
print("Train ∩ Test  :", len(train_set & test_set))
print("Valid ∩ Test  :", len(valid_set & test_set))

Train ∩ Valid : 378
Train ∩ Test  : 382
Valid ∩ Test  : 144


In [41]:
shared_all = train_set & valid_set & test_set

print("Shared across all splits:", len(shared_all))

Shared across all splits: 131
